> ## SOLUTION / ANSWER KEY &mdash; Lab 7.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-assemble-email-agent.ipynb`](../lab-11-assemble-email-agent.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.11 &mdash; Assemble the Email Agent

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Bind gather-only tools into an agent with create_agent
- Withhold send_email so the agent cannot auto-send
- Wrap the draft as needs_approval, with its tool trace

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, and the pipeline logic &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; The email-drafting agent, end to end](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-07-11"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Now assemble the email agent from Module 6's pieces (deck slides 14&ndash;16): `@tool` gather tools bound
with **`create_agent`** (a runnable `CompiledStateGraph`). The agent gathers (`lookup_order`,
`get_template`) then drafts a reply. The key design choice: the tools list is **gather-only** &mdash;
`send_email` is **not** bound &mdash; and the run's output is wrapped as a **`needs_approval`** draft.
The agent did the tedious 90%; the human keeps the send.

In [ ]:
from langchain_core.tools import tool

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

from langchain_core.tools import tool

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order's status, ETA and carrier by id."""
    return ORDERS.get(order_id, {"status": "unknown"})
@tool
def get_template(kind: str) -> str:
    """Fetch a reply template by kind."""
    return TEMPLATES.get(kind, "")
@tool
def send_email(to: str, body: str) -> str:
    """Send an email. (Defined to show the capability -- but DELIBERATELY WITHHELD from the agent.)"""
    return "SENT"
print("gather tools ready:", lookup_order.name, "&", get_template.name, "| withheld:", send_email.name)

## Your Turn
The guardrail here is what's **missing**: build the agent with **gather-only** tools (no `send_email`),
and wrap whatever it drafts as a **`needs_approval`** result. The draft text comes from the model at
run time (the optional cell), so the graded steps check the **wiring and the guardrail**, not the prose.

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

def gather_tools():
    return [lookup_order, get_template]

def make_email_agent():
    model = ChatOllama(model="llama3.2:1b")
    return create_agent(model, gather_tools())

def handle_email(draft, tools_used):
    # never auto-send: wrap the drafted reply as a result a human must approve
    return {"draft": draft, "status": "needs_approval", "tools_used": tools_used}

try:
    print('bound tools :', [t.name for t in gather_tools()])
    print('can it send?:', 'send_email' in [t.name for t in gather_tools()])
    print('agent type  :', type(make_email_agent()).__name__)
    demo = handle_email("Hi Priya, your order 4471 is due Friday.", ["lookup_order", "get_template"])
    print('wrapped     :', demo['status'], '| tools:', demo['tools_used'])
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the agent is a runnable CompiledStateGraph", lambda: type(make_email_agent()).__name__ == "CompiledStateGraph")
expect_true("it binds exactly the two gather tools", lambda: [t.name for t in gather_tools()] == ["lookup_order", "get_template"])
expect_true("the send tool is WITHHELD (the guardrail)", lambda: "send_email" not in [t.name for t in gather_tools()])
expect_true("send_email still EXISTS as a capability, just unbound", lambda: send_email.name == "send_email")
expect_true("a draft is wrapped as needs_approval, never sent", lambda: handle_email("d", [])["status"] == "needs_approval")
expect_true("the wrapper preserves the tool trace", lambda: handle_email("d", ["lookup_order"])["tools_used"] == ["lookup_order"])

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run the gather-only agent for real: it can look up and draft, but it has no way to send. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
from langchain_core.messages import AIMessage
def tools_used(messages):
    return [tc["name"] for m in messages for tc in (getattr(m, "tool_calls", None) or [])]
try:
    if ollama_up():
        agent = make_email_agent()
        result = agent.invoke({"messages": [{"role": "user",
                 "content": "Look up order 4471, then draft a one-line status reply. Do not send anything."}]},
                 config={"recursion_limit": 8})
        draft = result["messages"][-1].content
        out = handle_email(draft, tools_used(result["messages"]))
        print("tools used:", out["tools_used"])
        print("status    :", out["status"], "(the agent has no send tool, so it cannot auto-send)")
        print("draft     :", out["draft"])
    else:
        print("No Ollama reachable -- skipping the live run. The gather-only wiring above is what matters:")
        print("send_email is defined but never bound, so the agent gathers & drafts but cannot send.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
The guardrail is what's MISSING from the tools list -- send_email is never bound, so the agent gathers and drafts but cannot send. Next: run the whole pipeline over a suite.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>